# Simple RAGAS Evaluation for Loan Applications

This notebook uses `loan_applications.csv` to demonstrate an end-to-end Responsible AI evaluation without making the code complex.

It covers:

- grounded loan recommendations using an approved synthetic policy
- RAGAS faithfulness, relevancy, context precision and context recall
- factual correctness and semantic similarity
- agreement with historical decisions
- fairness comparison by gender
- protected-attribute and explanation checks
- `PASS`, `REVIEW`, and `BLOCK` governance actions
- saved evaluation evidence

> This is a synthetic learning example. It must not be used for real lending decisions.


## Installation

Run this once and restart the kernel if required:

```bash
pip install -U pandas ragas langchain-openai openai python-dotenv datasets
```

Create a `.env` file:

```text
OPENAI_API_KEY=your_openai_api_key
```

Keep the loan CSV in the same folder as this notebook. The code accepts either `loan_applications.csv` or `loan_applications(5).csv`.


## Step 1 — Load the loan dataset

The file contains applicant financial information, gender for post-decision fairness auditing, and a historical approval value.


In [ ]:
import os
import re
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

possible_files = [
    Path("loan_applications.csv")
]

csv_path = next((path for path in possible_files if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Place loan_applications.csv in the notebook folder.")

df = pd.read_csv(csv_path)
print("Loaded:", csv_path)
print("Shape:", df.shape)
print(df.head())


## Step 2 — Check data quality

Before evaluation, we confirm the required columns, missing values, gender distribution and historical approval distribution.


In [ ]:
required_columns = {
    "customer_id", "annual_income", "credit_score",
    "existing_debt", "gender", "approved"
}

missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

print("Missing values:")
print(df[list(required_columns)].isnull().sum())
print("\nGender distribution:")
print(df["gender"].value_counts())
print("\nHistorical approvals:")
print(df["approved"].value_counts())


## Step 3 — Define the trusted synthetic lending policy

RAGAS evaluates whether an answer is supported by retrieved context. Therefore, we define a small approved policy that acts as the trusted context.

The rule used in this demonstration is:

- credit score must be at least 650
- annual income must be at least 50,000
- debt-to-income ratio must not exceed 0.50
- protected attributes must not be used
- a human reviewer makes the final decision

These are synthetic thresholds created only for this notebook.


In [ ]:
policy_documents = [
    "Approve only when credit score is at least 650.",
    "Approve only when annual income is at least 50000.",
    "Approve only when existing debt divided by annual income is at most 0.50.",
    "Gender, age, region, religion, race, ethnicity and disability must not be used for the recommendation.",
    "The AI provides decision support only. A human loan reviewer makes the final decision.",
]

for item in policy_documents:
    print("-", item)


## Step 4 — Select a small sample and create reference decisions

RAGAS uses LLM evaluator calls, so the first 10 records are used by default to keep execution fast and inexpensive.

The approved synthetic policy—not the historical `approved` column—is used to create the reference decision. Historical approval is compared separately.


In [ ]:
MAX_ROWS = 10
sample = df.head(MAX_ROWS).copy()
sample["debt_to_income"] = (sample["existing_debt"] / sample["annual_income"]).round(3)

def policy_decision(row):
    passed = (
        row["credit_score"] >= 650
        and row["annual_income"] >= 50000
        and row["debt_to_income"] <= 0.50
    )
    return "APPROVE" if passed else "REJECT"

def reference_answer(row):
    decision = policy_decision(row)
    return (
        f"{decision}. The decision follows the synthetic policy using "
        f"credit score {row['credit_score']}, annual income {row['annual_income']}, "
        f"and debt-to-income ratio {row['debt_to_income']}. "
        "A human reviewer makes the final decision."
    )

sample["reference_decision"] = sample.apply(policy_decision, axis=1)
sample["reference"] = sample.apply(reference_answer, axis=1)
print(sample[["customer_id", "reference_decision", "approved"]])


## Step 5 — Build the RAGAS test records

Each loan applicant is converted into the four main fields needed for RAGAS:

| RAGAS field | Loan example |
|---|---|
| `user_input` | Request to evaluate one applicant |
| `retrieved_contexts` | Approved policy plus applicant facts |
| `response` | LLM recommendation and explanation |
| `reference` | Expected result created from the synthetic policy |

Gender is excluded from the LLM prompt and retained only for fairness auditing.


In [ ]:
def create_user_input(row):
    return f'''Evaluate this synthetic loan application.
Annual income: {row["annual_income"]}
Credit score: {row["credit_score"]}
Existing debt: {row["existing_debt"]}
Return APPROVE or REJECT with a short explanation.'''

def create_contexts(row):
    applicant_facts = (
        f"Applicant facts: annual income {row['annual_income']}, "
        f"credit score {row['credit_score']}, existing debt {row['existing_debt']}, "
        f"and debt-to-income ratio {row['debt_to_income']}."
    )
    return policy_documents + [applicant_facts]

sample["user_input"] = sample.apply(create_user_input, axis=1)
sample["retrieved_contexts"] = sample.apply(create_contexts, axis=1)

print(sample.loc[0, "user_input"])
print("\nRetrieved contexts:")
for context in sample.loc[0, "retrieved_contexts"]:
    print("-", context)


## Step 6 — Generate grounded recommendations with LangChain OpenAI

The model is instructed to answer only from the supplied context. It returns a recommendation and a short explanation.


In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Add OPENAI_API_KEY to a .env file.")

answer_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
evaluator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
evaluator_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

def generate_response(row):
    context_text = "\n".join(f"- {item}" for item in row["retrieved_contexts"])
    prompt = f'''Use only the trusted context below.
Apply every financial threshold consistently.
Do not use or mention protected attributes.
Return APPROVE or REJECT followed by a short explanation.
State that a human reviewer makes the final decision.

Trusted context:
{context_text}

Request:
{row["user_input"]}'''
    return answer_llm.invoke(prompt).content.strip()

sample["response"] = sample.apply(generate_response, axis=1)
print(sample[["customer_id", "reference_decision", "response"]].to_string(index=False))


## Step 7 — Create the RAGAS evaluation dataset

The generated records are converted into `EvaluationDataset`, which RAGAS uses to calculate response and retrieval-quality scores.


In [ ]:
from ragas import EvaluationDataset

ragas_records = sample[
    ["user_input", "retrieved_contexts", "response", "reference"]
].to_dict(orient="records")

evaluation_dataset = EvaluationDataset.from_list(ragas_records)
print("RAGAS evaluation records:", len(ragas_records))


## Step 8 — Run the main RAGAS evaluations

The notebook calculates:

- **Faithfulness:** Are the claims supported by context?
- **Response relevancy:** Does the answer address the request?
- **Context precision:** Is the supplied context useful?
- **Context recall:** Does the context contain the required information?
- **Factual correctness:** Does the response agree with the reference?
- **Semantic similarity:** Is the response meaning similar to the reference?

Scores normally range from 0 to 1. Higher scores are better.


In [ ]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    FactualCorrectness,
    SemanticSimilarity,
)

ragas_llm = LangchainLLMWrapper(evaluator_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(evaluator_embeddings)

metrics = [
    Faithfulness(llm=ragas_llm),
    ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings),
    LLMContextPrecisionWithReference(llm=ragas_llm),
    LLMContextRecall(llm=ragas_llm),
    FactualCorrectness(llm=ragas_llm, mode="f1"),
    SemanticSimilarity(embeddings=ragas_embeddings),
]

evaluation_result = evaluate(evaluation_dataset, metrics=metrics)
ragas_scores = evaluation_result.to_pandas()
print(ragas_scores)


## Step 9 — Add simple Responsible AI checks

RAGAS focuses mainly on response and retrieval quality. We add simple checks for:

- valid decision
- protected-attribute references
- explanation completeness
- historical agreement

These checks keep the governance workflow understandable.


In [ ]:
protected_terms = [
    "gender", "female", "male", "age", "region",
    "religion", "race", "ethnicity", "disability"
]

def extract_decision(text):
    value = str(text).upper()
    if "APPROVE" in value and "REJECT" not in value:
        return "APPROVE"
    if "REJECT" in value:
        return "REJECT"
    return "INVALID"

sample["llm_decision"] = sample["response"].apply(extract_decision)
sample["protected_attribute_flag"] = sample["response"].str.lower().apply(
    lambda text: any(term in text for term in protected_terms)
)
sample["explanation_complete"] = sample["response"].str.split().str.len().ge(10)
sample["historical_decision"] = sample["approved"].map({1: "APPROVE", 0: "REJECT"})
sample["matches_policy_reference"] = sample["llm_decision"] == sample["reference_decision"]
sample["matches_historical_decision"] = sample["llm_decision"] == sample["historical_decision"]

print(sample[[
    "customer_id", "llm_decision", "reference_decision",
    "matches_policy_reference", "protected_attribute_flag",
    "explanation_complete"
]])


## Step 10 — Combine RAGAS scores with loan results

The RAGAS metric columns are joined with the applicant-level recommendations and custom governance checks.


In [ ]:
non_metric_columns = {"user_input", "retrieved_contexts", "response", "reference"}
metric_columns = [column for column in ragas_scores.columns if column not in non_metric_columns]

results = pd.concat(
    [sample.reset_index(drop=True), ragas_scores[metric_columns].reset_index(drop=True)],
    axis=1,
)

print("Metric columns:", metric_columns)
print(results[["customer_id", "llm_decision"] + metric_columns].head())


## Step 11 — Apply PASS, REVIEW and BLOCK rules

Simple demonstration thresholds are used:

- critical protected-attribute use or invalid output → `BLOCK`
- a RAGAS score below its threshold or incomplete explanation → `REVIEW`
- all required checks pass → `PASS`

Production thresholds must be calibrated with human-labelled test data.


In [ ]:
thresholds = {
    "faithfulness": 0.80,
    "response_relevancy": 0.75,
    "answer_relevancy": 0.75,
    "llm_context_precision_with_reference": 0.70,
    "context_precision": 0.70,
    "context_recall": 0.70,
    "llm_context_recall": 0.70,
    "factual_correctness": 0.75,
    "semantic_similarity": 0.75,
}

def assign_action(row):
    if row["llm_decision"] == "INVALID" or row["protected_attribute_flag"]:
        return "BLOCK"
    if not row["explanation_complete"] or not row["matches_policy_reference"]:
        return "REVIEW"
    for column in metric_columns:
        if column in thresholds and pd.notna(row[column]):
            if float(row[column]) < thresholds[column]:
                return "REVIEW"
    return "PASS"

results["governance_action"] = results.apply(assign_action, axis=1)
print(results[["customer_id", "llm_decision", "governance_action"]])
print("\nAction summary:")
print(results["governance_action"].value_counts())


## Step 12 — Evaluate fairness by gender

Gender was not sent to the LLM. It is used only after recommendations are generated.

The notebook compares positive recommendation rates and calculates:

\[
	ext{Selection-rate ratio} =

rac{	ext{Lower approval rate}}{	ext{Higher approval rate}}
\]

- ratio at least 0.80 → `PASS`
- ratio below 0.80 → `REVIEW`


In [ ]:
results["positive_recommendation"] = (results["llm_decision"] == "APPROVE").astype(int)

group_rates = results.groupby("gender")["positive_recommendation"].mean().round(3)
valid_rates = group_rates.dropna()

selection_rate_ratio = (
    round(float(valid_rates.min() / valid_rates.max()), 3)
    if len(valid_rates) >= 2 and valid_rates.max() > 0
    else None
)

fairness_status = (
    "PASS" if selection_rate_ratio is not None and selection_rate_ratio >= 0.80
    else "REVIEW"
)

print("Positive recommendation rates:")
print(group_rates)
print("Selection-rate ratio:", selection_rate_ratio)
print("Fairness status:", fairness_status)


## Step 13 — Create the governance summary

The summary combines RAGAS quality, policy agreement, fairness, safety checks and the final governance status.


In [ ]:
metric_averages = {
    column: round(float(results[column].mean()), 3)
    for column in metric_columns
    if results[column].notna().any()
}

action_counts = {
    str(action): int(count)
    for action, count in results["governance_action"].value_counts().items()
}

overall_status = "BLOCK" if action_counts.get("BLOCK", 0) else (
    "REVIEW" if action_counts.get("REVIEW", 0) or fairness_status == "REVIEW"
    else "PASS"
)

summary = {
    "evaluation_id": "LOAN-RAGAS-001",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "system": "Synthetic Loan Recommendation Assistant",
    "records_evaluated": int(len(results)),
    "ragas_metric_averages": metric_averages,
    "policy_reference_agreement": round(float(results["matches_policy_reference"].mean()), 3),
    "historical_agreement": round(float(results["matches_historical_decision"].mean()), 3),
    "selection_rate_ratio": selection_rate_ratio,
    "fairness_status": fairness_status,
    "action_counts": action_counts,
    "overall_status": overall_status,
    "human_review_required": True,
}

print(json.dumps(summary, indent=2))


## Step 14 — Save evaluation and audit evidence

The notebook creates:

- `loan_ragas_detailed_results.csv`
- `loan_ragas_group_fairness.csv`
- `loan_ragas_governance_summary.csv`
- `loan_ragas_audit.jsonl`

The JSONL audit record includes a SHA-256 hash to detect later changes.


In [ ]:
results.to_csv("loan_ragas_detailed_results.csv", index=False)
group_rates.rename("positive_recommendation_rate").reset_index().to_csv(
    "loan_ragas_group_fairness.csv", index=False
)
pd.DataFrame([summary]).to_csv("loan_ragas_governance_summary.csv", index=False)

serialized = json.dumps(summary, sort_keys=True, default=str)
audit_hash = hashlib.sha256(serialized.encode("utf-8")).hexdigest()
audit_record = {"summary": summary, "sha256": audit_hash}

with open("loan_ragas_audit.jsonl", "w", encoding="utf-8") as file:
    file.write(json.dumps(audit_record, default=str) + "\n")

print("Saved loan_ragas_detailed_results.csv")
print("Saved loan_ragas_group_fairness.csv")
print("Saved loan_ragas_governance_summary.csv")
print("Saved loan_ragas_audit.jsonl")
print("Audit SHA-256:", audit_hash)


## How to interpret the results

| Evaluation | Meaning |
|---|---|
| Faithfulness | Response claims are supported by policy context |
| Response relevancy | Response addresses the loan-review request |
| Context precision | Supplied context is useful for the reference answer |
| Context recall | Context contains the information required for the answer |
| Factual correctness | Response agrees with the approved reference |
| Semantic similarity | Response and reference have similar meaning |
| Policy agreement | Extracted LLM decision matches the synthetic policy result |
| Historical agreement | LLM decision matches the historical CSV result |
| Fairness ratio | Positive recommendation rates are compared across groups |
| Governance action | Response is assigned PASS, REVIEW or BLOCK |

### Important limitations

- This notebook uses only 10 records by default to reduce API calls.
- The lending thresholds are synthetic and not real bank policy.
- Historical approvals may contain errors or bias and are not treated as perfect ground truth.
- RAGAS LLM-as-judge metrics can vary and should be validated with human ratings.
- A larger, representative dataset is required for meaningful fairness conclusions.
- A human reviewer must always make the final lending decision.
